# EHR Copilot — ORPO v4 Training on Colab

**Goal:** Fine-tune Qwen2.5-Coder-7B-Instruct with ORPO (Odds-Ratio Preference Optimization) to improve SQL generation and abstention on the EHRSQL MIMIC-III benchmark.

**Target:** RS(10) ≥ 0.813 (EHRSQL 2024 leaderboard top score — LG AI / KAIST)

**Recommended runtime:** A100 40GB (Runtime → Change runtime type → A100)

---

## Files to upload (from the `colab/` folder in the repo)

| File | Size | Description |
|------|------|-------------|
| `data.zip` | 37 MB | ORPO pairs, EHRSQL data, bge-large embeddings, template classifier |
| `src.zip` | 0.2 MB | Python source package |
| `checkpoint_orpo_v3.z00` | 95 MB | Checkpoint part 1/4 |
| `checkpoint_orpo_v3.z01` | 95 MB | Checkpoint part 2/4 |
| `checkpoint_orpo_v3.z02` | 95 MB | Checkpoint part 3/4 |
| `checkpoint_orpo_v3.z03` | 3 MB  | Checkpoint part 4/4 |

## Steps
1. ▶ **Cell 1** — check GPU
2. ▶ **Cell 2** — install dependencies (~3 min)
3. ▶ **Cell 3** — mount Google Drive
4. ▶ **Cell 4** — upload all 6 files above
5. ▶ **Cell 5** — reassemble checkpoint + extract all zips + verify
6. ▶ **Cell 6** — detect GPU and set batch config
7. ▶ **Cell 7** — run ORPO v4 training (6–8 hrs A100, 24–36 hrs T4)
8. ▶ **Cell 8** — save adapter to Google Drive
9. ▶ **Cell 9** *(optional)* — run RAGAS retrieval eval
10. ▶ **Cell 10** *(optional)* — run full eval + RS(10) score


In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess, torch

r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                    '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Install dependencies (~3 min) ─────────────────────────────────
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q \
    "trl>=0.9.0" "peft>=0.11.0" "bitsandbytes>=0.43.0" \
    "accelerate>=0.30.0" "transformers>=4.45.0" \
    "datasets>=2.19.0" "huggingface-hub>=0.23.0" \
    "sentence-transformers>=2.7.0" "rank-bm25>=0.2.2" \
    "scikit-learn>=1.4.0" "joblib>=1.3.0"
print('All dependencies installed.')

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/EHRCopilot'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
print(f'Drive ready: {DRIVE_DIR}')

In [ ]:
# ── Cell 4: Upload files ───────────────────────────────────────────────────
# Upload ALL 6 files at once:
#   data.zip
#   src.zip
#   checkpoint_orpo_v3.z00  (95 MB)
#   checkpoint_orpo_v3.z01  (95 MB)
#   checkpoint_orpo_v3.z02  (95 MB)
#   checkpoint_orpo_v3.z03  (3 MB)

from google.colab import files
print('Select all 6 files and upload...')
uploaded = files.upload()
print(f'\nUploaded {len(uploaded)} file(s):')
for name in sorted(uploaded):
    print(f'  {name}  ({len(uploaded[name])/1024/1024:.1f} MB)')

In [ ]:
# ── Cell 5: Reassemble checkpoint parts + extract all zips ─────────────────
import zipfile, os, glob

os.chdir('/content')

# Reassemble split checkpoint into checkpoint_orpo_v3.zip
parts = sorted(glob.glob('checkpoint_orpo_v3.z*'))
if parts:
    print(f'Reassembling {len(parts)} checkpoint parts ...')
    with open('checkpoint_orpo_v3.zip', 'wb') as out:
        for p in parts:
            with open(p, 'rb') as f:
                out.write(f.read())
            os.remove(p)
    size_mb = os.path.getsize('checkpoint_orpo_v3.zip') / 1024 / 1024
    print(f'  checkpoint_orpo_v3.zip  ({size_mb:.1f} MB)')
else:
    print('WARNING: No checkpoint parts found (checkpoint_orpo_v3.z*)')

# Extract all zips
for zname in ['data.zip', 'checkpoint_orpo_v3.zip', 'src.zip']:
    if os.path.exists(zname):
        print(f'Extracting {zname} ...')
        with zipfile.ZipFile(zname) as zf:
            zf.extractall('.')
        os.remove(zname)
        print(f'  done')
    else:
        print(f'  WARNING: {zname} not found')

# Verify
checks = [
    'data/ehrsql/orpo_v4_pairs.jsonl',
    'data/ehrsql/template_classifier.pkl',
    'data/ehrsql/train_embeddings_bge_large.npy',
    'data/ehrsql/ehrsql/mimic_iii/train.json',
    'checkpoints/orpo_v3/adapter_final/adapter_model.safetensors',
    'src/ehrcopilot/finetune/abstention_dpo.py',
]
print('\nFile check:')
all_ok = True
for p in checks:
    ok = os.path.exists(p)
    print(f'  {"OK" if ok else "MISSING"}  {p}')
    if not ok:
        all_ok = False
print('\nReady to train!' if all_ok else '\nSome files missing — recheck uploads.')

In [ ]:
# ── Cell 6: Detect GPU and set batch config ────────────────────────────────
import os, sys, torch

os.chdir('/content')
sys.path.insert(0, '/content/src')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)')

if vram_gb >= 35:    # A100 40GB
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 4, 4, 1536
elif vram_gb >= 20:  # A100 20GB / L4
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 2, 8, 1536
else:                # T4 16GB
    BATCH_SIZE, GRAD_ACCUM, MAX_LENGTH = 1, 16, 1024

print(f'batch={BATCH_SIZE}  grad_accum={GRAD_ACCUM}  max_length={MAX_LENGTH}')
print(f'Effective batch size: {BATCH_SIZE * GRAD_ACCUM}  (target: 16)')

In [ ]:
# ── Cell 7: Run ORPO v4 training ───────────────────────────────────────────
#
# Starting from ORPO v3 adapter (SFT + v1-v3 preference trained).
# 5,218 pairs: 4,856 SQL quality + 362 abstention.
# Loss target: < 0.05  |  Accuracy target: > 75%  |  Margin: consistently positive
#
# A100 40GB: ~6-8 hrs  |  T4 16GB: ~24-36 hrs

import subprocess, sys

cmd = [
    sys.executable, '-m', 'ehrcopilot.finetune.abstention_dpo',
    '--pairs',      'data/ehrsql/orpo_v4_pairs.jsonl',
    '--adapter',    'checkpoints/orpo_v3/adapter_final',
    '--output',     'checkpoints/orpo_v4',
    '--epochs',     '1',
    '--lr',         '5e-6',
    '--max-length', str(MAX_LENGTH),
]

print('Command:', ' '.join(cmd))
print('-' * 60)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    env={**os.environ, 'PYTHONPATH': '/content/src', 'TOKENIZERS_PARALLELISM': 'false'},
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f'\nExit code: {proc.returncode}')

In [ ]:
# ── Cell 8: Save adapter to Google Drive ───────────────────────────────────
import shutil, os

SRC = '/content/checkpoints/orpo_v4'
DST = '/content/drive/MyDrive/EHRCopilot/checkpoints/orpo_v4'

if os.path.exists(SRC):
    if os.path.exists(DST):
        shutil.rmtree(DST)
    shutil.copytree(SRC, DST)
    print(f'Adapter saved to Google Drive: {DST}')
    for f in os.listdir(f'{DST}/adapter_final'):
        print(f'  {f}')
else:
    print('No checkpoint at', SRC, '— did training complete?')

In [ ]:
# ── Cell 9 (optional): RAGAS retrieval eval ────────────────────────────────
# Confirms template retrieval quality: target Recall@2 >= 90%, Precision@2 >= 90%.
# Expected: ~92.7% on both (pre-validated locally).

import subprocess, sys

result = subprocess.run([
    sys.executable, '-m', 'ehrcopilot.eval.rag_eval',
    '--train',             'data/ehrsql/ehrsql/mimic_iii/train.json',
    '--test',              'data/ehrsql/ehrsql/mimic_iii/test.json',
    '--mode',              'template',
    '--embed-cache',       'data/ehrsql/train_embeddings_bge_large.npy',
    '--classifier-cache',  'data/ehrsql/template_classifier.pkl',
    '--output',            'rag_eval_results.json',
    '--k',                 '1,2,5',
], capture_output=True, text=True,
   env={**os.environ, 'PYTHONPATH': '/content/src'})

print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])

In [ ]:
# ── Cell 10 (optional): Full eval — RS(N) scoring ─────────────────────────
# SQL generation → execution-guided repair → RS(0)/RS(5)/RS(10) scoring.
# Uses template retrieval (92.7% precision) for 2-shot examples.
# A100: ~1.5-2 hrs  |  T4: ~4-6 hrs

import subprocess, sys, json, os

ADAPTER = 'checkpoints/orpo_v4/adapter_final'
OUTPUT  = 'orpo_v4_eval_results.json'

proc = subprocess.Popen([
    sys.executable, '-m', 'ehrcopilot.eval.harness',
    'data/ehrsql/ehrsql/mimic_iii/test.json',
    '--model',            ADAPTER,
    '--output',           OUTPUT,
    '--repair',
    '--few-shot',         'data/ehrsql/ehrsql/mimic_iii/train.json',
    '--retrieval-mode',   'template',
    '--classifier-cache', 'data/ehrsql/template_classifier.pkl',
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
   env={**os.environ, 'PYTHONPATH': '/content/src', 'TOKENIZERS_PARALLELISM': 'false'})

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()

if os.path.exists(OUTPUT):
    r = json.load(open(OUTPUT))
    print('\n' + '='*50)
    print(f'  EX:     {r.get("EX", 0):.4f}')
    print(f'  RS(0):  {r.get("RS(0)", 0):.4f}')
    print(f'  RS(10): {r.get("RS(10)", 0):.4f}   (target ≥ 0.813)')
    print('='*50)